In [10]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import itertools as it
import sympy as sp
import numpy as np
import flint
from algebra import SparAlg
from families import family
from equations import EquationSystem

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
def reflection(d):
    def refl(a, b): return (b, a)
    return dict(name='reflection', block=refl, mid=None)

def relabel(perm):
    def permute(a, b): return (perm[a], perm[b])
    d = len(perm)
    Pm = np.zeros((d, d))
    for c in range(d):
        Pm[perm[c], c] = 1
    return dict(name=f'relabel{tuple(perm)}', block=permute, mid=Pm)


In [7]:
perm = [4, 3, 5, 2, 1, 0, 7, 6]
relab_dict = relabel(perm)
print(relab_dict)

{'name': 'relabel(4, 3, 5, 2, 1, 0, 7, 6)', 'block': <function relabel.<locals>.permute at 0x107e7f880>, 'mid': array([[0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 1., 0.]])}


In [ ]:
class Solver:
    def __init__(self, ansatz, htilde=None, symmetries=(), extra_equations=(), verbose=True):
        self.ans = ansatz
        self.d = ansatz.d
        self.es = EquationSystem(ansatz, htilde=htilde)
        self.symmetries = list(symmetries)
        self.results = {}
        self.verbose = verbose
        self.log(f'symmetries of system {self.ans.name}: {self.symmetries}')

    def log(self, a):
        if self.verbose:
            print(f'[Solver] {a}', flush=True)

    def independent_linear(self, F):
        if not F: return []
        n = self.es.ctx.nvars()
        M = flint.fmpq_mat(len(F), n+1)     #+1 for the constant term
        for i, p in enumerate(F):
            for m, c in zip(p.monoms(), p.coeffs()):
                j = m.index(1) if sum(m) == 1 else n
                M[i, j] = c
        R, r = M.rref()
        out = []
        for i in range(r):
            dct = dict()
            for j in range(n):
                if R[i, j] != 0:
                    e = [0]*n
                    e[j] = 1
                    dct[tuple(e)] = R[i, j]
            if R[i, n] != 0:
                dct[(0,)*n] = R[i, n]
            out.append(self.es.ctx.from_dict(dct))
        return out

    def forms(self):
        es, A, d = self.es, self.es.alg, self.d
        h = es.density()
        self.block_forms = {}
        for (a, b) in it.product(range(d), repeat=2):
            F = []
            for (rw, cl), (re, im) in h.items():
                if rw[0] == a and rw[2] == b and rw[1] < cl[1]:
                    F += [p for p in (re, im) if not p.is_zero()]
            self.block_forms[(a, b)] = self.independent_linear(F)
        comps = A.canonical(h, 3)
        self.two_local_forms = [p for (rw, cl), v in comps[3].items() for p in v if not p.is_zero()]
        q = A.comm(A.pad(h, 0, 1), A.pad(h, 1, 0))
        comps_q = A.canonical(q, 4)
        # comps = {l: {(rw, cl): (re, im)}}, c is itself a dict
        self.vacuous_forms = [p for c in comps_q.values() for v in c.values() for p in v if not p.is_zero()]

    def generate(self, linear_reduce=True):
        eqs = self.es.generate()
        if linear_reduce:
            self.es.linear_reduce()
        self.forms()
        return self.es.equations

    def active_candidates(self):
        return [k for k in it.product(range(self.d), repeat=2) if self.block_forms[k]]

    def consistent(self, S):
        def key(k):
            return tuple(sorted(map(str, self.block_forms[k])))
        act = {key(k) for k in S}
        return all(key(k) not in act for k in self.active_candidates() if k not in S)

    def orbit(self, S):
        orbit, frontier = {S}, [S]
        while frontier:
            X = frontier.pop()
            for g in self.symmetries:
                Y = frozenset(g['block'](*k) for k in X)
                self.log(f'{set(X)} --> {set(Y)} by symmetry {g['name']}')
                if Y not in orbit:
                    self.log(f'{set(Y)} was not in orbit')
                    orbit.add(Y)
                    frontier.append(Y)
                else:
                    self.log(f'{set(Y)} already in orbit')
        return orbit

    def patterns(self, min_active=1, max_active=None):
        cands = self.active_candidates()
        max_active = max_active or len(cands)
        seen, reps = set(), []
        for r in range(min_active, max_active+1):
            for S in it.combinations(cands, r):
                S = frozenset(S)
                if not self.consistent(S): continue
                if S in seen: continue
                orbit = self.orbit(S)
                seen |= orbit
                reps.append((tuple(sorted(S)), len(orbit)))
        self.log(f'{len(reps)} unique activity patterns from {len(cands)} activatable blocks')
        return reps

    def stratum(self, S):
        S = sorted(S)
        inactive = [k for k in self.active_candidates() if k not in S]
        lin = [p for k in inactive for p in self.block_forms[k]]
        first = self.block_forms[S[0]]
        # we don't use k == S[0] so that the code instantly filters for redundancies in the active blocks (if block == first, then we can't use it for saturation later)
        others = [self.block_forms[k] for k in S if not self.block_forms[k] != first]
        charts = []
        for c in range(len(first)):
            eqs = lin + first[:c] + [first[c]-1]
            charts.append(dict(label=f'S={S} chart={c}', extra=eqs, saturate=others))
    

In [ ]:
ans = family('B_RZ2')
eqsys = EquationSystem(ans, htilde=None, q5_form='irf', order='degrevlex')
A = eqsys.alg
symmetries = [reflection(3), relabel([0, 2, 1])]
sol = Solver(ans, symmetries=symmetries, verbose=False)
sol.forms()

def lin_forms(neq):
    nvar = sol.es.nvars
    rand_arr = np.random.randint(-10, 10, (neq, nvar))
    F = []
    for i in range(neq):
        rand_dict = dict()
        for j in range(nvar):
            k = [0]*nvar
            k[j] = 1
            rand_dict[tuple(k)] = flint.fmpz(int(rand_arr[i, j]))
        fi = sol.es.ctx.from_dict(rand_dict)
        F.append(fi)
    return F

F = lin_forms(16)
Fl = sol.independent_linear(F)
print(len(F))
print(len(Fl))

16
10


In [197]:
act_cands = sol.active_candidates()
S = {(0, 1)}
orbit = sol.orbit(frozenset(S))
print(f'orbit of {S}: {[set(X) for X in orbit]}')
patterns = sol.patterns()
print([len(i[0]) for i in patterns])

orbit of {(0, 1)}: [{(0, 1)}, {(0, 2)}, {(1, 0)}, {(2, 0)}]
[1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 9]
